# Day 5.4 — Permissions, Approval and Limits
The model proposes; Python disposes. That single sentence is the safety model of this
course, and this lesson is where it becomes running code.

We will watch an external action pause, a rejection end cleanly, a destructive tool be
refused even when it is allow-listed, an unrecognised risk level fail closed, and a
runaway loop get stopped by a limit the configuration cannot raise.


## Before you begin

### Learning outcomes

- Turn a tool's risk level into an allow / approval / deny decision.
- Pause a run on a checkpoint and resolve it in both directions.
- Show that limits and unknown inputs both fail closed.

Architecture reference: [Day 5 diagrams D16](../diagrams/source/day_05.md).

### Expected observation

`send_email` pauses at `pending_approval`; rejecting produces the status `cancelled`; `erase_workspace` is denied while allow-listed; a config asking for 500 steps is held to the runtime's cap.


## Concept briefing

## Permissions, approval and limits

The model proposes an action; Python decides whether it may happen. That decision
uses two independent facts: is the tool on this agent's allow-list, and what is
the tool's local **risk level**? The mini harness maps risk to a decision in one
place, `policy.RISK_POLICY`:

| Risk | Meaning | Decision |
|---|---|---|
| `read` | no effect outside the process | allow |
| `write` | reversible local change | allow |
| `external` | leaves the machine or is visible to others | approval |
| `destructive` | irreversible | deny |

Anything not in that table returns `deny`. That is **failing closed**: an unknown
or misspelled risk label must never be read as permission. A policy that raises
an exception on an unfamiliar input is worse, because a crash in the wrong place
can be caught and ignored, while an explicit `deny` is a decision that gets logged.

`approval` is not a question asked in the conversation. The run stops, the exact
pending action is written to a checkpoint, and a separate `resume` call carries
the human answer. Rejection is a **successful** safety outcome, so it has its own
status, `cancelled`, distinct from `failed`.

Limits are the other half of control. A configuration may request any number of
steps, but the runtime owns a hard ceiling (`MAX_STEPS_HARD_CAP`); the effective
limit is the smaller of the two, and that is the number the events report. A loop
that ends at `step_limit` has not crashed - it has been stopped on purpose.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — The whole policy, printed

`policy.decide` is under ten lines. Print the table it uses before trusting anything else.


In [ ]:
from mini_harness import RISK_POLICY, build_demo_registry, decide

print("risk level  -> decision")
for risk, decision in RISK_POLICY.items():
    print(f"  {risk:<12}-> {decision}")

print()
registry = build_demo_registry()
task = load_config("task_agent")
print("For task_agent, tool by tool:")
for spec in registry.discover():
    on_list = spec.name in task.allowed_tools
    print(f"  {spec.name:<16} risk={spec.risk:<12} allow-listed={str(on_list):<5} "
          f"decision={decide(task, spec)}")
print()
print("Note create_draft and send_email are both allow-listed, and get different answers.")

## Step 2 — An external action pauses the run

`send_email` is classified `external`. The run does not fail and does not ask the model
for permission. It stops, and writes down exactly what it wanted to do.


In [ ]:
from mini_harness import HarnessRuntime, MockModel

runtime = HarnessRuntime(build_demo_registry(), MockModel())
pending = runtime.run(task, "Send a synthetic course update")

print("Status        :", pending.status)
print("Paused tool   :", pending.pending_action["tool"])
print("Exact arguments the human is being asked to approve:")
for key, value in pending.pending_action["arguments"].items():
    print(f"    {key}: {value}")
print()
print("Saved to the checkpoint store under run id", pending.run_id)
print("Checkpoint keys:", sorted(runtime.checkpoints.load(pending.run_id)))

## Step 3 — Rejecting is a success, and it has its own status

Read the status carefully. A rejected run is **`cancelled`**, not `failed`. Nothing went
wrong; a person said no, and the harness records that as a distinct outcome.


In [ ]:
rejected = runtime.resume(pending.run_id, task, approved=False)

print("Status after rejection:", rejected.status)
print("Message               :", rejected.output)
print()
print("Is that the same as a failure?", rejected.status == "failed")
print("The five end states a run can reach are:")
print("  completed | pending_approval | cancelled | failed | step_limit")
print()
print("Last three events:")
for event in rejected.events[-3:]:
    print(f"  {event['event']:<20}",
          {k: v for k, v in event["details"].items() if k != "history"})
print()
print("Checkpoint after resolving:", runtime.checkpoints.load(pending.run_id))
print("-> cleared, so the same action cannot be approved twice.")

## Step 4 — Allow-listing a destructive tool changes nothing

A common misunderstanding is that the allow-list *is* the permission system. It is only
half of it: the tool's risk level is the other half, and `destructive` always loses.


In [ ]:
greedy = load_config("task_agent")
greedy.allowed_tools.append("erase_workspace")     # deliberately over-permissive config

destructive = registry.get("erase_workspace").spec
print("erase_workspace now on the allow-list:", "erase_workspace" in greedy.allowed_tools)
print("It is therefore discoverable        :",
      "erase_workspace" in [s.name for s in registry.discover(greedy.allowed_tools)])
print("Policy decision                     :", decide(greedy, destructive))
print()
# Now make the agent actually ask for it, so we see what the RUN does, not just
# what the policy function returns. (mock_plan is how we steer the mock model.)
greedy.mock_plan = [{"tool": "erase_workspace", "arguments": {}}]
result = runtime.run(greedy, "Erase everything")
print("A run that genuinely requests erase_workspace:")
print("  status:", result.status)
print("  reason:", result.output)
print("  events:", [e["event"] for e in result.events])
print()
print("The tool function never ran. 'deny' stops the run before execution,")
print("and the denial itself is recorded as a policy_decision event.")

## Step 5 — An unrecognised risk level fails closed

New tools arrive from places you do not control (Day 5.6 imports them over MCP). What if
one carries a risk label the policy table has never seen?


In [ ]:
from mini_harness import AgentConfig, ToolSpec

# A tool whose risk label is not one of read/write/external/destructive.
mystery = ToolSpec("mystery_action", "A capability from somewhere else",
                   {"type": "object", "properties": {}}, "quantum")
somebody = AgentConfig("somebody", "Try anything.", ["mystery_action"])

print("Is 'quantum' in the policy table?", "quantum" in RISK_POLICY)
print("Decision for an unknown risk    :", decide(somebody, mystery))
print()
print("It denies. It does not raise, and it certainly does not allow.")
print("That is 'failing closed': the safe answer for an input we do not understand.")
print("An exception here would be worse - exceptions get caught and swallowed,")
print("while a 'deny' is a decision that gets recorded in the event log.")

## Step 6 — Limits the configuration cannot raise

A loop that never terminates is not a hypothetical. Here is a model that always asks for
the same tool again, and two different ways the harness stops it.


In [ ]:
from mini_harness import MAX_STEPS_HARD_CAP, ModelDecision, effective_step_limit

class EndlessModel:
    """Always asks for the same tool. Never says 'done'."""
    def decide(self, prompt, config, tools, history):
        return ModelDecision("tool", tool="lookup_notes", arguments={"query": prompt})

print("Hard cap compiled into runtime.py:", MAX_STEPS_HARD_CAP)
print()

for requested in (2, 500):
    config = load_config("research_agent")
    config.max_steps = requested
    limit = effective_step_limit(config)
    print(f"config asks for {requested:>3} steps -> effective limit {limit}")

print()
looping = load_config("research_agent")
looping.max_steps = 2
outcome = HarnessRuntime(build_demo_registry(), EndlessModel()).run(looping, "keep going")
print("Status      :", outcome.status)
print("First event :", outcome.events[0]["event"], outcome.events[0]["details"])
print("Last event  :", outcome.events[-1]["event"], outcome.events[-1]["details"])
print()
print("The event reports the EFFECTIVE limit, so the log never disagrees with reality.")

### Try it yourself

Step 3 rejected the pending email. Predict what changes in the event trace if you approve
it instead — and what the final status becomes.


In [ ]:
# --- Worked solution ---
# Start a fresh run, because the earlier checkpoint was consumed by the rejection.
approved_run = runtime.run(task, "Send a synthetic course update")
print("Paused again at:", approved_run.status, "->", approved_run.pending_action["tool"])

# resume() carries the human answer. The model is NOT asked again: the exact
# arguments come back out of the checkpoint, not out of a new model call.
final = runtime.resume(approved_run.run_id, task, approved=True)

print()
print("Final status:", final.status)
print("Tool output :", final.output)
print()
print("Event trace, rejection vs approval:")
print("  rejected :", [e["event"] for e in rejected.events[-3:]])
print("  approved :", [e["event"] for e in final.events[-4:]])
print()
print("Both paths record approval_resolved. Only the approved one reaches tool_completed,")
print("and only then does the side effect happen.")

### Checkpoint

**1. Why must the approval decision come back through `resume()` rather than by asking the model again?**

<details><summary>Show answer</summary>

Because the model would be re-deciding, not confirming. The checkpoint holds the exact arguments a human reviewed. Re-prompting could produce a different recipient or body, and the human would have approved something that never ran.

</details>

**2. A tool arrives from an outside server carrying `risk="maybe"`. What happens?**

<details><summary>Show answer</summary>

`decide` returns `deny`, because `RISK_POLICY.get(risk, 'deny')` falls back to denial for anything it does not recognise. An unknown label can never be read as permission, and the denial is recorded as a normal policy decision rather than as a crash.

</details>

### Recap

- Limitation: an allow-list on its own cannot express 'visible, but not without a human', and a config could otherwise grant itself unlimited steps.
- Layer added: a risk-to-decision table that fails closed, a checkpointed approval pause, and a hard step cap owned by the runtime.
- Evidence: `send_email` paused and resolved to `cancelled` then `completed`; an allow-listed `erase_workspace` was still denied; an unknown risk denied; a 500-step request was held to 10.
